# 09c — Retrieval-Augmented LLM Prompting (GPT-NER style)

Variant of Notebook 09 that replaces *fixed* in-context demonstrations with **kNN-retrieved**
ones: for each test sentence we embed it (MiniLM) and pull the `RETRIEVE_K` most semantically
similar examples from the same budget-K labeled pool, following GPT-NER (Wang et al. 2023) and
the demonstration-selection idea in FsPONER (Tang et al. 2024). This tests whether *which* demos
you show matters, holding the labeled budget fixed.

Notes:
- Retrieval makes each prompt unique, so OpenAI's automatic prefix caching no longer applies —
  expect somewhat higher cost/latency than 09 (mitigated by using only `RETRIEVE_K` demos rather
  than all of them). Consider a `EVAL_LIMIT = 50` smoke test first.
- Results go to a **separate** `results/llm_retrieval_results.csv` (method `llm_retrieval`), so
  the original Arm 3 numbers are untouched and directly comparable.

# 09 - LLM Few-Shot Prompting (Arm 3)

Third adaptation strategy: **no gradient updates at all.** For each (target dataset, budget) we
build a prompt whose in-context demonstrations are the *same* labeled examples the fine-tuning
arms trained on, then ask an LLM to tag each **full test-set** sentence. Predictions are scored
with the **same `compute_entity_metrics`** as every other arm (typed micro-F1, boundary F1,
per-type), so Arm 3 is directly comparable to Arms 1 and 2.

Two deliberate design choices:

1. **One demo seed (42).** Fine-tuning on 50 examples is high-variance, so Arms 1-2 run all
   three seeds. Running the LLM over three seeds would triple the token cost for a variance
   estimate we can partly infer from the fine-tuning arms, so this arm is a single-seed point
   estimate, flagged as such in the analysis.
2. **Provider + fallback.** Default is the **OpenAI API** (`gpt-4o-mini`) when `OPENAI_API_KEY`
   is set; otherwise the notebook falls back to a small **free local** open-weights model run
   through `transformers`. The key is read from the environment / Colab secrets, never
   hardcoded. The provider and model are recorded in every results row, since a small local
   model is a much weaker "LLM arm" than a hosted model and the two are not interchangeable.

OpenAI automatically caches the long, repeated demonstration prefix, so most input tokens after
the first call in a configuration are billed at the discounted cached rate. Per-sentence
predictions are cached to disk for resume, and token usage / USD is tracked per call.

In [9]:
!pip -q install "seqeval==1.2.2" openai pandas sentence-transformers

## Step 1 — Mount Drive and configure

In [10]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import json
from pathlib import Path

PROCESSED      = Path('/content/drive/MyDrive/AAI590/data/processed')
LABELS_DIR     = PROCESSED / 'label_maps'
MODELS_DIR     = PROCESSED / 'models'
RESULTS_DIR    = PROCESSED / 'results'
FEWSHOT_SPLITS = PROCESSED / 'fewshot_splits'

TARGET_DATASETS = ['wnut17', 'scierc']
BUDGETS = [50, 100, 200]
SEEDS   = [13, 42, 101]

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

print('processed dir :', PROCESSED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
processed dir : /content/drive/MyDrive/AAI590/data/processed


In [ ]:
import os, re, time
import pandas as pd

results_dir = RESULTS_DIR / 'llm_prompting'
results_dir.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR = results_dir / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

# --- OpenAI API key --- set it ONE of these ways (checked in order):
#   1) paste it into OPENAI_API_KEY below TEMPORARILY (clear it before you commit/push!),
#   2) a Colab secret named OPENAI_API_KEY, 3) an already-set env var, or
#   4) leave everything blank -> you'll be prompted for it securely (hidden input).
OPENAI_API_KEY = ""   # e.g. "sk-proj-..."  -- SAFEST is to leave this "" and use the prompt

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if not os.environ.get('OPENAI_API_KEY'):
    try:
        from google.colab import userdata
        if userdata.get('OPENAI_API_KEY'):
            os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception:
        pass
if not os.environ.get('OPENAI_API_KEY'):
    try:
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OPENAI_API_KEY (hidden): ')
    except Exception:
        pass

DEMO_SEED = 42            # single fixed seed for in-context demos (see above)
EVAL_LIMIT = None         # None = full test set; set an int (e.g. 50) for a cheap smoke test

# OpenAI when a key is present; otherwise the free local open-weights fallback
PROVIDER = 'openai' if os.environ.get('OPENAI_API_KEY') else 'local'
OPENAI_MODEL = 'gpt-4o-mini'   # cheap + capable; switch to 'gpt-4o' / 'gpt-4.1-mini' for higher quality
LOCAL_MODEL  = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_OUTPUT_TOKENS = 512

# USD per million tokens for the OpenAI model -- VERIFY against current pricing
PRICE_PER_MTOK = {'input': 0.15, 'output': 0.60, 'cache_read': 0.075}   # gpt-4o-mini

model_name = OPENAI_MODEL if PROVIDER == 'openai' else LOCAL_MODEL
RESULTS_CSV = results_dir / 'llm_retrieval_results.csv'
print(f'provider: {PROVIDER} | model: {model_name} | demo seed: {DEMO_SEED} | eval limit: {EVAL_LIMIT}')
if PROVIDER == 'local':
    print('NOTE: no OPENAI_API_KEY found -- using the free local fallback model (slower, weaker).')

provider: openai | model: gpt-4o-mini | demo seed: 42 | eval limit: None


## Step 2 — Metrics (identical to Notebooks 05 / 08)

In [12]:
from seqeval.metrics import classification_report

def collapse_to_boundary(tag):
    if tag == 'O':
        return 'O'
    return ('B-ENT' if tag.startswith('B-') else 'I-ENT')

def compute_entity_metrics(true_tags, pred_tags):
    """Typed + boundary micro F1 and a per-type breakdown, all via seqeval default mode,
    matching Notebooks 05/08 so every arm's F1 is computed identically."""
    rep = classification_report(true_tags, pred_tags, output_dict=True, zero_division=0)
    micro = rep['micro avg']
    per_type = {}
    for k, v in rep.items():
        if k in ('micro avg', 'macro avg', 'weighted avg'):
            continue
        per_type[k] = {'precision': float(v['precision']), 'recall': float(v['recall']),
                       'f1': float(v['f1-score']), 'support': int(v['support'])}
    tb = [[collapse_to_boundary(t) for t in s] for s in true_tags]
    pb = [[collapse_to_boundary(t) for t in s] for s in pred_tags]
    brep = classification_report(tb, pb, output_dict=True, zero_division=0)['micro avg']
    return {
        'typed_precision': float(micro['precision']),
        'typed_recall': float(micro['recall']),
        'typed_f1': float(micro['f1-score']),
        'boundary_f1': float(brep['f1-score']),
        'support': int(micro['support']),
        'per_type': per_type,
    }

## Step 3 — Prompt construction and JSON→BIO alignment

Each prompt is: task instructions (with the dataset's own entity types), the *k* labeled
demonstrations, then the test sentence. The model answers with a JSON list of
`{"text": ..., "type": ...}` entities, which we align back onto the token sequence to produce
BIO tags (first not-yet-tagged contiguous span, case-insensitive fallback). JSON output is far
easier to parse robustly than one-tag-per-token; parse failures and unalignable entities are
counted and reported.

In [13]:
from seqeval.metrics.sequence_labeling import get_entities

TYPE_NOTES = {
    'wnut17': ('person, location, corporation, product, creative-work (songs, movies, '
               'books...), group (bands, sports teams...)'),
    'scierc': ('Task, Method, Metric, Material, Generic (a generic term like "approach" '
               'referring to a specific one), OtherScientificTerm'),
}

def entity_types_of(dataset_name):
    train_full = load_jsonl(PROCESSED / dataset_name / f'{dataset_name}_train.jsonl')
    return sorted({t.split('-', 1)[1] for r in train_full for t in r['tags'] if t != 'O'})

def gold_entities(tokens, tags):
    return [{'text': ' '.join(tokens[s:e + 1]), 'type': etype}
            for etype, s, e in get_entities(tags)]

def format_example(tokens, tags=None):
    line = 'Sentence: ' + ' '.join(tokens)
    return line + '\nEntities:' if tags is None else line + '\nEntities: ' + json.dumps(gold_entities(tokens, tags))

def build_system_prompt(dataset_name, demo_rows):
    types = entity_types_of(dataset_name)
    parts = [
        'You are a named entity recognition tagger.',
        f"Entity types for this domain: {', '.join(types)}.",
        f"Type hints: {TYPE_NOTES[dataset_name]}.",
        'For the given sentence, list every entity as a JSON array of objects with keys '
        '"text" (the exact contiguous span, copied verbatim from the sentence) and "type" '
        '(one of the types above). Preserve order of appearance. If there are no entities, '
        'answer []. Answer with ONLY the JSON array, nothing else.',
    ]
    if demo_rows:
        parts += ['', f'Here are {len(demo_rows)} labeled examples from this domain:', '']
        parts += [format_example(r['tokens'], r['tags']) for r in demo_rows]
    return '\n'.join(parts)

def parse_entities(text):
    try:
        out = json.loads(text)
        return out if isinstance(out, list) else None
    except json.JSONDecodeError:
        pass
    m = re.search(r'\[.*\]', text, re.DOTALL)
    if m:
        try:
            out = json.loads(m.group(0))
            return out if isinstance(out, list) else None
        except json.JSONDecodeError:
            return None
    return None

def entities_to_bio(tokens, entities, allowed_types):
    tags = ['O'] * len(tokens)
    dropped = 0
    lower = [t.lower() for t in tokens]
    canonical = {t.lower(): t for t in allowed_types}
    for ent in entities or []:
        if not isinstance(ent, dict) or 'text' not in ent or 'type' not in ent:
            dropped += 1; continue
        etype = canonical.get(str(ent['type']).lower())
        if etype is None:
            dropped += 1; continue          # hallucinated type
        span = str(ent['text']).split()
        if not span:
            dropped += 1; continue
        placed = False
        for exact in (True, False):
            hay = tokens if exact else lower
            needle = span if exact else [w.lower() for w in span]
            for i in range(len(tokens) - len(span) + 1):
                if hay[i:i + len(span)] == needle and all(t == 'O' for t in tags[i:i + len(span)]):
                    tags[i] = f'B-{etype}'
                    for k in range(i + 1, i + len(span)):
                        tags[k] = f'I-{etype}'
                    placed = True; break
            if placed:
                break
        if not placed:
            dropped += 1                     # text not found as a contiguous span
    return tags, dropped

## Step 4 — Provider call functions

In [14]:
# provider call functions -- both return (response_text, info_dict)

if PROVIDER == 'openai':
    import openai
    from openai import OpenAI
    client = OpenAI()

    def _create_with_retry(**kwargs):
        # Handle transient 429 rate-limit (tokens-per-minute) and connection blips by backing
        # off and retrying, instead of crashing the run. `insufficient_quota` is NOT retried --
        # that means the account is out of credit and waiting won't help.
        delay = 1.0
        for _ in range(8):
            try:
                return client.chat.completions.create(**kwargs)
            except openai.RateLimitError as e:
                if 'insufficient_quota' in str(e):
                    raise
                time.sleep(delay); delay = min(30.0, delay * 2)
            except (openai.APITimeoutError, openai.APIConnectionError, openai.InternalServerError):
                time.sleep(delay); delay = min(30.0, delay * 2)
        return client.chat.completions.create(**kwargs)  # final attempt; let it raise if still failing

    def call_llm(system_prompt, user_prompt):
        # OpenAI automatically caches long repeated prefixes (the demo block) and reports the
        # cached portion in usage.prompt_tokens_details.cached_tokens.
        t0 = time.time()
        resp = _create_with_retry(
            model=OPENAI_MODEL, max_tokens=MAX_OUTPUT_TOKENS, temperature=0,
            messages=[{'role': 'system', 'content': system_prompt},
                      {'role': 'user', 'content': user_prompt}],
        )
        u = resp.usage
        details = getattr(u, 'prompt_tokens_details', None)
        cached = (getattr(details, 'cached_tokens', 0) or 0) if details else 0
        uncached_in = max(0, u.prompt_tokens - cached)
        cost = (uncached_in * PRICE_PER_MTOK['input'] + cached * PRICE_PER_MTOK['cache_read']
                + u.completion_tokens * PRICE_PER_MTOK['output']) / 1e6
        text = resp.choices[0].message.content or ""
        return text, {'input_tokens': u.prompt_tokens, 'output_tokens': u.completion_tokens,
                      'cache_read_tokens': cached, 'cache_write_tokens': 0,
                      'cost_usd': cost, 'seconds': time.time() - t0}

else:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    llm_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL)
    llm_model = AutoModelForCausalLM.from_pretrained(
        LOCAL_MODEL, torch_dtype=torch.float16 if device == 'cuda' else torch.float32).to(device).eval()
    print('loaded', LOCAL_MODEL, 'on', device)

    def call_llm(system_prompt, user_prompt):
        t0 = time.time()
        chat = [{'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt}]
        enc = llm_tokenizer.apply_chat_template(chat, add_generation_prompt=True,
                                                return_tensors='pt', return_dict=True).to(device)
        n_in = enc['input_ids'].shape[1]
        with torch.no_grad():
            out = llm_model.generate(**enc, max_new_tokens=MAX_OUTPUT_TOKENS, do_sample=False,
                                     pad_token_id=llm_tokenizer.eos_token_id)
        text = llm_tokenizer.decode(out[0][n_in:], skip_special_tokens=True)
        return text, {'input_tokens': int(n_in), 'output_tokens': int(out.shape[1] - n_in),
                      'cache_read_tokens': 0, 'cache_write_tokens': 0,
                      'cost_usd': 0.0, 'seconds': time.time() - t0}

## Step 5 — Run the grid (6 configs; per-sentence caching for resume)

In [15]:
# --- sentence embedder for kNN demonstration retrieval (GPT-NER, Wang et al. 2023) ---
from sentence_transformers import SentenceTransformer
import numpy as np

RETRIEVE_K = 15   # demonstrations retrieved per test sentence, from the budget-K labeled pool
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('embedder ready; retrieving top', RETRIEVE_K, 'demos per test sentence')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedder ready; retrieving top 15 demos per test sentence


In [16]:
grid = [(d, b) for d in TARGET_DATASETS for b in BUDGETS]

done_configs = set()
if RESULTS_CSV.exists():
    prev = pd.read_csv(RESULTS_CSV)
    done_configs = set(zip(prev['dataset'], prev['budget'], prev['provider']))
    print(f'found {len(done_configs)} finished configs')

for ds_name, budget in grid:
    if (ds_name, budget, PROVIDER) in done_configs:
        print('skipping', (ds_name, budget, PROVIDER)); continue
    print(f'\n=========== {ds_name} budget={budget} ({PROVIDER}, retrieval k={RETRIEVE_K}) ===========')

    demo_pool = load_jsonl(FEWSHOT_SPLITS / ds_name / f'{ds_name}_train_{budget}_seed_{DEMO_SEED}.jsonl')
    allowed = entity_types_of(ds_name)
    demo_texts = [' '.join(r['tokens']) for r in demo_pool]
    demo_emb = embedder.encode(demo_texts, convert_to_numpy=True, normalize_embeddings=True)
    k = min(RETRIEVE_K, len(demo_pool))

    eval_rows = load_jsonl(PROCESSED / ds_name / f'{ds_name}_test.jsonl')
    if EVAL_LIMIT:
        eval_rows = eval_rows[:EVAL_LIMIT]

    pred_fp = PREDICTIONS_DIR / f'retr_{PROVIDER}_{ds_name}_{budget}.jsonl'
    cached = {}
    if pred_fp.exists():
        for r in load_jsonl(pred_fp):
            cached[r['id']] = r
        print(f'resuming: {len(cached)} sentences already predicted')

    t_start = time.time()
    parse_failures = dropped_entities = 0
    totals = {'input_tokens': 0, 'output_tokens': 0, 'cache_read_tokens': 0,
              'cache_write_tokens': 0, 'cost_usd': 0.0}
    gold_tags, pred_tags = [], []

    for i, row in enumerate(eval_rows):
        if row['id'] in cached:
            rec = cached[row['id']]
        else:
            # retrieve the k most similar labeled demos for THIS sentence
            q = embedder.encode([' '.join(row['tokens'])], convert_to_numpy=True,
                                normalize_embeddings=True)[0]
            top = (demo_emb @ q).argsort()[::-1][:k]
            retrieved = [demo_pool[j] for j in top]
            system_prompt = build_system_prompt(ds_name, retrieved)
            raw, info = call_llm(system_prompt, format_example(row['tokens']))
            ents = parse_entities(raw)
            tags, dropped = entities_to_bio(row['tokens'], ents, allowed)
            rec = {'id': row['id'], 'raw_response': raw, 'pred_tags': tags,
                   'parse_failed': ents is None, 'dropped_entities': dropped, **info}
            with open(pred_fp, 'a', encoding='utf-8') as f:
                f.write(json.dumps(rec, ensure_ascii=True) + '\n')
        parse_failures += int(rec['parse_failed'])
        dropped_entities += rec['dropped_entities']
        for kk in totals:
            totals[kk] += rec.get(kk, 0)
        gold_tags.append(row['tags']); pred_tags.append(rec['pred_tags'])
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(eval_rows)} sentences (${totals['cost_usd']:.3f} so far)")

    m = compute_entity_metrics(gold_tags, pred_tags)
    wall = time.time() - t_start
    row_out = {
        'method': 'llm_retrieval', 'dataset': ds_name, 'budget': budget, 'seed': DEMO_SEED,
        'provider': PROVIDER, 'model': model_name, 'retrieve_k': RETRIEVE_K,
        'test_typed_precision': m['typed_precision'], 'test_typed_recall': m['typed_recall'],
        'test_typed_f1': m['typed_f1'], 'test_boundary_f1': m['boundary_f1'],
        'support': m['support'], 'n_eval': len(eval_rows),
        'parse_failures': parse_failures, 'dropped_entities': dropped_entities,
        'wall_seconds': round(wall, 1), 'cost_usd': round(totals['cost_usd'], 4),
        'input_tokens': totals['input_tokens'], 'output_tokens': totals['output_tokens'],
        'cache_read_tokens': totals['cache_read_tokens'], 'cache_write_tokens': totals['cache_write_tokens'],
        'per_type': json.dumps(m['per_type']),
    }
    pd.DataFrame([row_out]).to_csv(RESULTS_CSV, mode='a', index=False, header=not RESULTS_CSV.exists())
    print(f"{ds_name} b={budget}: typed F1={m['typed_f1']:.3f} boundary F1={m['boundary_f1']:.3f} "
          f"cost=${totals['cost_usd']:.3f} in {wall:.0f}s, {parse_failures} parse failures")

print('\nall configurations finished')


=========== wnut17 budget=50 (openai, retrieval k=15) ===========
  50/1287 sentences ($0.008 so far)
  100/1287 sentences ($0.015 so far)
  150/1287 sentences ($0.023 so far)
  200/1287 sentences ($0.030 so far)
  250/1287 sentences ($0.038 so far)
  300/1287 sentences ($0.046 so far)
  350/1287 sentences ($0.053 so far)
  400/1287 sentences ($0.060 so far)
  450/1287 sentences ($0.068 so far)
  500/1287 sentences ($0.075 so far)
  550/1287 sentences ($0.083 so far)
  600/1287 sentences ($0.090 so far)
  650/1287 sentences ($0.097 so far)
  700/1287 sentences ($0.105 so far)
  750/1287 sentences ($0.112 so far)
  800/1287 sentences ($0.119 so far)
  850/1287 sentences ($0.126 so far)
  900/1287 sentences ($0.133 so far)
  950/1287 sentences ($0.141 so far)
  1000/1287 sentences ($0.148 so far)
  1050/1287 sentences ($0.155 so far)
  1100/1287 sentences ($0.163 so far)
  1150/1287 sentences ($0.170 so far)
  1200/1287 sentences ($0.177 so far)
  1250/1287 sentences ($0.184 so far)
wnu

In [17]:
results = pd.read_csv(RESULTS_CSV)
cols = ['dataset', 'budget', 'provider', 'model', 'test_typed_f1', 'test_boundary_f1',
        'cost_usd', 'wall_seconds', 'parse_failures', 'n_eval']
display(results[cols].sort_values(['dataset', 'budget']).round(3))

,dataset,budget,provider,model,test_typed_f1,test_boundary_f1,cost_usd,wall_seconds,parse_failures,n_eval
3,scierc,50,openai,gpt-4o-mini,0.421,0.600,0.113,594.7,0,551
4,scierc,100,openai,gpt-4o-mini,0.421,0.594,0.119,590.9,0,551
5,scierc,200,openai,gpt-4o-mini,0.445,0.606,0.126,605.4,0,551
0,wnut17,50,openai,gpt-4o-mini,0.505,0.584,0.189,1078.5,0,1287
1,wnut17,100,openai,gpt-4o-mini,0.522,0.591,0.189,1039.2,0,1287
2,wnut17,200,openai,gpt-4o-mini,0.519,0.592,0.196,1058.2,0,1287
